In [2]:
# ============================================================================
# 00b_prepare_datasets.ipynb
# ----------------------------------------------------------------------------
# Beyond Full-Schema Prompting: A Graph-based Semantic Layer for Text-to-SQL
#
# Purpose:
#   Download Spider + BIRD (mini-dev) and convert them into notebook 01's layout:
#       data/<Dkey>/questions.json    [{qid, db_id, question, gold_sql,
#                                       gold_tables, gold_columns, difficulty?}]
#       data/<Dkey>/schema.json       [{db_id, tables, foreign_keys,
#                                       glossary, join_patterns}, ...]
#       data/<Dkey>/db/<db_id>.sqlite executable databases for EX scoring
#
#   Key design (v3): gold_tables / gold_columns are derived by parsing each gold
#   SQL with SQLGlot AND QUALIFYING it against the real schema, so bare columns
#   like 'age' become 'singer.age'. This makes strict table.column schema-linking
#   metrics correct (the earlier version left bare/empty columns). Join patterns
#   are auto-mined from the qualified SQLs (paper Section 10.2).
#
#   Source reality:
#       Spider questions : HF "xlangai/spider" (parquet; questions + gold SQL)
#       Spider schema+DB : HF "HAL-9001/spider-databases" (spider_data.zip)
#       BIRD questions   : HF "birdsql/bird_mini_dev" split "mini_dev_sqlite"
#       BIRD schema      : HF "567-labs/bird-dev-tables" (dev_tables.json)
#       BIRD databases   : NOT on HF -> place minidev SQLite DBs manually
#
#   Anonymity: no filesystem paths are printed; only counts/summaries are shown.
# ============================================================================


# %% [markdown]
# ## Cell 1 - Load shared configuration

# %%
%run 00_setup_and_config.ipynb


# %%
# One-time installs if missing (uncomment on first run):
# !pip install datasets huggingface_hub sqlglot

import re
import shutil
import sqlite3
import zipfile
from collections import defaultdict

import sqlglot
from sqlglot import exp
from sqlglot.optimizer.qualify import qualify

from huggingface_hub import hf_hub_download

print("Notebook 00b dependencies ready.")


# %% [markdown]
# ## Cell 2 - Source configuration
# See header for what each source provides. Only the BIRD SQLite databases need
# manual placement:
#   data/_raw/bird/dev_databases/<db_id>/<db_id>.sqlite

# %%
RAW_DIR = DATA_DIR / "_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

SPIDER_SPLIT = "validation"          # HF split for D1 questions ("validation" = dev)
BIRD_SPLIT = "mini_dev_sqlite"       # BIRD SQLite-dialect split
BIRD_FINANCE_DB = "financial"        # D2_fin proxy database id

SPIDER_DB_REPO = "HAL-9001/spider-databases"    # spider_data.zip (DBs + tables.json)
BIRD_TABLES_REPO = "567-labs/bird-dev-tables"   # dev_tables.json


# %% [markdown]
# ## Cell 3 - Schema converters (Spider/BIRD tables.json -> schema.json dicts)
# Handles BIRD composite primary keys (list entries). Sample values are pulled
# from the actual SQLite DB when present.

# %%
def _sample_values_from_db(sqlite_path, table, column, k=3):
    try:
        con = sqlite3.connect(str(sqlite_path))
        con.text_factory = lambda b: b.decode("utf-8", "ignore")
        cur = con.cursor()
        cur.execute(f'SELECT DISTINCT "{column}" FROM "{table}" '
                    f'WHERE "{column}" IS NOT NULL LIMIT {k}')
        vals = [r[0] for r in cur.fetchall()]
        con.close()
        return vals
    except Exception:
        return []


def tables_json_to_schema(tables_json_obj, db_dir):
    """
    Convert a Spider/BIRD tables.json (index-encoded) into schema.json dicts.
    Composite primary keys (BIRD, list entries) are flattened.
    """
    out = []
    for entry in tables_json_obj:
        db_id = entry["db_id"]
        table_names = entry["table_names_original"]
        col_defs = entry["column_names_original"]      # [[tbl_idx, col_name], ...]
        col_types = entry["column_types"]

        pk_idx = set()
        for pk in entry.get("primary_keys", []):
            if isinstance(pk, list):
                pk_idx.update(pk)
            else:
                pk_idx.add(pk)

        fk_pairs = entry.get("foreign_keys", [])       # [[col_i, col_j], ...]
        sqlite_path = db_dir / db_id / f"{db_id}.sqlite"

        tables = {t: {"columns": {}} for t in table_names}
        for ci, (ti, cname) in enumerate(col_defs):
            if ti < 0:
                continue
            tname = table_names[ti]
            samples = _sample_values_from_db(sqlite_path, tname, cname) \
                if sqlite_path.exists() else []
            tables[tname]["columns"][cname] = {
                "type": col_types[ci].upper() if ci < len(col_types) else "TEXT",
                "pk": ci in pk_idx,
                "desc": "",
                "sample_values": samples,
            }

        foreign_keys = []
        for a, b in fk_pairs:
            ta, ca = col_defs[a]
            tb, cb = col_defs[b]
            foreign_keys.append({"from": f"{table_names[ta]}.{ca}",
                                 "to": f"{table_names[tb]}.{cb}"})

        out.append({
            "db_id": db_id, "tables": tables, "foreign_keys": foreign_keys,
            "glossary": [], "join_patterns": [],
        })
    return out


# %% [markdown]
# ## Cell 4 - SQL analysis with schema-qualified columns
# Build a sqlglot schema dict {table: {column: type}} per db, then qualify each
# gold SQL so every column carries its table. Extract gold_tables/gold_columns and
# join conditions from the QUALIFIED tree. Falls back gracefully on parse errors.

# %%
def _sqlglot_schema(schema_entry):
    """{table: {column: type}} in the shape sqlglot.qualify expects."""
    sch = {}
    for tname, tinfo in schema_entry["tables"].items():
        sch[tname] = {c: (ci.get("type") or "TEXT")
                      for c, ci in tinfo["columns"].items()}
    return sch


def _strip(s):
    return (s or "").replace('"', "").replace("`", "").replace("[", "").replace("]", "")


def analyze_sql_qualified(sql, schema_entry):
    """
    Return (tables, columns, joins) with columns fully qualified as 'table.column'
    (lowercased) using the schema. If qualification fails, fall back to a
    best-effort parse that keeps whatever qualification exists.
    """
    tables, columns, joins = set(), set(), []
    sch = _sqlglot_schema(schema_entry)

    # Parse.
    try:
        tree = sqlglot.parse_one(sql, read="sqlite")
    except Exception:
        return tables, columns, joins

    # Try to fully qualify columns against the schema (dialect-aware, case-insensitive).
    qualified = None
    try:
        qualified = qualify(tree.copy(), schema=sch, dialect="sqlite",
                            qualify_columns=True, validate_qualify_columns=False,
                            identify=False)
    except Exception:
        qualified = tree  # fall back to the raw tree

    # tables (from the raw tree; qualify may rewrite/alias)
    alias2table = {}
    for tnode in tree.find_all(exp.Table):
        tname = _strip(tnode.name)
        if not tname:
            continue
        tables.add(tname.lower())
        alias = _strip(tnode.alias_or_name)
        if alias:
            alias2table[alias.lower()] = tname

    # columns (from the qualified tree)
    for cnode in qualified.find_all(exp.Column):
        col = _strip(cnode.name)
        tbl = _strip(cnode.table)
        if not col:
            continue
        if tbl:
            real = alias2table.get(tbl.lower(), tbl)
            columns.add(f"{real.lower()}.{col.lower()}")
        else:
            # Still unqualified (e.g. ambiguous or schema miss): keep bare, lower.
            columns.add(col.lower())

    # join equalities from the qualified tree
    for eq in qualified.find_all(exp.EQ):
        l, r = eq.left, eq.right
        if isinstance(l, exp.Column) and isinstance(r, exp.Column):
            def q(c):
                t = _strip(c.table)
                real = alias2table.get(t.lower(), t)
                return f"{real.lower()}.{_strip(c.name).lower()}" if real \
                    else _strip(c.name).lower()
            joins.append(f"{q(l)}={q(r)}")

    return tables, columns, joins


def mine_join_patterns(rows_for_db, schema_entry):
    """Frequency-ranked join patterns from qualified gold SQLs (paper 10.2)."""
    freq = defaultdict(int)
    for q in rows_for_db:
        _, _, joins = analyze_sql_qualified(q["gold_sql"], schema_entry)
        for j in set(joins):
            freq[j] += 1
    patterns = []
    for cond, f in sorted(freq.items(), key=lambda x: x[1], reverse=True):
        try:
            a, b = cond.split("=")
            path = [a.split(".")[0], b.split(".")[0]]
        except Exception:
            path = []
        patterns.append({"path": path, "on": [cond], "freq": f})
    return patterns


# %% [markdown]
# ## Cell 5 - Downloaders (Spider fully automatic; BIRD DBs manual)

# %%
USE_HF = True


def _ensure_spider_data():
    """Download+unzip spider_data.zip once. Returns (tables_path, db_root)."""
    spider_raw = RAW_DIR / "spider"
    spider_raw.mkdir(parents=True, exist_ok=True)
    unpacked = spider_raw / "spider_data"
    tables_path = unpacked / "tables.json"
    db_root = unpacked / "database"
    if not tables_path.exists() or not db_root.exists():
        zip_path = hf_hub_download(repo_id=SPIDER_DB_REPO,
                                   filename="spider_data.zip", repo_type="dataset")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(spider_raw)
    return tables_path, db_root


def download_spider():
    from datasets import load_dataset
    ds = load_dataset("xlangai/spider", split=SPIDER_SPLIT)
    questions_raw = [dict(x) for x in ds]
    tables_path, db_root = _ensure_spider_data()
    tables_raw = json.loads(tables_path.read_text(encoding="utf-8")) \
        if tables_path.exists() else []
    referenced = sorted(set(q["db_id"] for q in questions_raw))
    present = sum((db_root / d / f"{d}.sqlite").exists() for d in referenced)
    print(f"  [spider] schema tables={len(tables_raw)}, "
          f"sqlite present/needed={present}/{len(referenced)}")
    return questions_raw, tables_raw, db_root


def download_bird():
    from datasets import load_dataset
    ds = load_dataset("birdsql/bird_mini_dev", split=BIRD_SPLIT)
    questions_raw = [dict(x) for x in ds]
    bird_raw = RAW_DIR / "bird"
    bird_raw.mkdir(parents=True, exist_ok=True)
    db_root = bird_raw / "dev_databases"
    tables_path = bird_raw / "dev_tables.json"
    if not tables_path.exists():
        try:
            src = hf_hub_download(repo_id=BIRD_TABLES_REPO,
                                  filename="dev_tables.json", repo_type="dataset")
            shutil.copy(src, tables_path)
        except Exception:
            pass
    tables_raw = json.loads(tables_path.read_text(encoding="utf-8")) \
        if tables_path.exists() else []
    n_db = len(list(db_root.glob("*/*.sqlite"))) if db_root.exists() else 0
    if n_db == 0:
        print(f"  [bird] schema tables={len(tables_raw)}, sqlite present=0 "
              f"-> place minidev SQLite DBs under data/_raw/bird/dev_databases/ "
              f"to enable EX scoring.")
    else:
        print(f"  [bird] schema tables={len(tables_raw)}, sqlite present={n_db}")
    return questions_raw, tables_raw, db_root


# %% [markdown]
# ## Cell 6 - Normalizers (schema-first: qualify columns against the schema)
# Requires schema_by_db so gold columns can be qualified. Produces the unified
# question rows with fully-qualified gold_tables/gold_columns.

# %%
def normalize_questions(questions_raw, schema_by_db, source):
    """
    source: 'spider' or 'bird'. Returns unified rows with qualified gold columns.
    """
    rows = []
    for i, q in enumerate(questions_raw):
        db_id = q.get("db_id") or q.get("database_id") or ""
        if source == "spider":
            gold_sql = q.get("query") or q.get("SQL") or ""
            qid = f"spider_{SPIDER_SPLIT}_{i:05d}"
            extra = {"difficulty": q.get("hardness", "")}
        else:
            gold_sql = q.get("SQL") or q.get("sql") or q.get("query") or ""
            qid = f"bird_{i:05d}"
            extra = {"difficulty": q.get("difficulty", ""),
                     "evidence": q.get("evidence", "")}

        schema_entry = schema_by_db.get(db_id)
        if schema_entry is not None:
            tset, cset, _ = analyze_sql_qualified(gold_sql, schema_entry)
        else:
            tset, cset = set(), set()

        rows.append({
            "qid": qid, "db_id": db_id, "question": q.get("question", ""),
            "gold_sql": gold_sql,
            "gold_tables": sorted(tset),
            "gold_columns": sorted(cset),
            **extra,
        })
    return rows


# %% [markdown]
# ## Cell 7 - Writer (mines join patterns, copies DBs, writes json)

# %%
def write_dataset(dkey, questions_rows, schema_list, db_root, restrict_db=None):
    out_dir = DATA_DIR / dkey
    (out_dir / "db").mkdir(parents=True, exist_ok=True)

    if restrict_db:
        schema_list = [s for s in schema_list if s["db_id"] == restrict_db]
        questions_rows = [q for q in questions_rows if q["db_id"] == restrict_db]

    schema_by_db = {s["db_id"]: s for s in schema_list}

    # Mine join patterns per db from qualified gold SQLs.
    q_by_db = defaultdict(list)
    for q in questions_rows:
        q_by_db[q["db_id"]].append(q)
    for db_id, s in schema_by_db.items():
        s["join_patterns"] = mine_join_patterns(q_by_db.get(db_id, []), s)

    # Copy referenced SQLite DBs.
    copied = 0
    if db_root is not None:
        for db_id in set(q["db_id"] for q in questions_rows):
            src = Path(db_root) / db_id / f"{db_id}.sqlite"
            if src.exists():
                shutil.copy(src, out_dir / "db" / f"{db_id}.sqlite")
                copied += 1

    (out_dir / "schema.json").write_text(
        json.dumps(list(schema_by_db.values()), ensure_ascii=False, indent=2),
        encoding="utf-8")
    (out_dir / "questions.json").write_text(
        json.dumps(questions_rows, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[{dkey}] wrote {len(questions_rows)} questions across "
          f"{len(schema_by_db)} databases; copied {copied} sqlite DB(s).")


# %% [markdown]
# ## Cell 8 - Build D1 (Spider)

# %%
BUILD_SPIDER = True

if BUILD_SPIDER:
    if USE_HF:
        q_raw, tables_raw, db_root = download_spider()
    else:
        tables_path, db_root = _ensure_spider_data()
        tables_raw = json.loads(tables_path.read_text(encoding="utf-8"))
        q_raw = json.loads((RAW_DIR / "spider" / "dev.json").read_text(encoding="utf-8"))

    spider_schema = tables_json_to_schema(tables_raw, Path(db_root)) if tables_raw else []
    schema_by_db = {s["db_id"]: s for s in spider_schema}
    q_rows = normalize_questions(q_raw, schema_by_db, source="spider")
    write_dataset("D1", q_rows, spider_schema, db_root)
else:
    print("[D1] skipped.")


# %% [markdown]
# ## Cell 9 - Build D2 (BIRD) and D2_fin (financial only)
# Questions + schema build automatically; SQLite DBs are copied from
# data/_raw/bird/dev_databases/ (staged by Cell 9b) into data/D2/db/.

# %%
BUILD_BIRD = True

if BUILD_BIRD:
    if USE_HF:
        q_raw, tables_raw, db_root = download_bird()
    else:
        tables_raw = json.loads((RAW_DIR / "bird" / "dev_tables.json").read_text(encoding="utf-8"))
        qfile = next((RAW_DIR / "bird").glob("*sqlite*json"), None) or \
                next((RAW_DIR / "bird").glob("*dev*json"), None)
        q_raw = json.loads(Path(qfile).read_text(encoding="utf-8"))
        db_root = RAW_DIR / "bird" / "dev_databases"

    bird_schema = tables_json_to_schema(tables_raw, Path(db_root)) if tables_raw else []
    schema_by_db = {s["db_id"]: s for s in bird_schema}
    q_rows = normalize_questions(q_raw, schema_by_db, source="bird")

    if bird_schema:
        write_dataset("D2", q_rows, bird_schema, db_root)
        write_dataset("D2_fin", q_rows, bird_schema, db_root, restrict_db=BIRD_FINANCE_DB)
    else:
        (DATA_DIR / "D2").mkdir(parents=True, exist_ok=True)
        (DATA_DIR / "D2" / "questions.json").write_text(
            json.dumps(q_rows, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"[D2] wrote {len(q_rows)} questions only (schema/DB pending).")
else:
    print("[D2] skipped.")

# %% [markdown]
# ## Cell 9b - (manual) Stage BIRD SQLite DBs from the extracted MINIDEV folder
# The SQLite databases live under the MINIDEV folder (not _mysql/_postgresql).
# We read the authoritative db_id list from dev_tables.json, then search the
# MINIDEV tree for each <db_id>.sqlite and copy it into the expected location:
#   data/_raw/bird/dev_databases/<db_id>/<db_id>.sqlite
# Anonymity: no paths or database names are printed - only counts.

# %%
import shutil

# Root that contains the SQLite version of mini-dev (has dev_databases inside).
# Kept relative to the project so no absolute path is exposed in output.
MINIDEV_SQLITE_ROOT = RAW_DIR / "minidev" / "minidev" / "MINIDEV"

# Authoritative BIRD db_id list (from the tables file staged earlier).
_bird_tables_path = RAW_DIR / "bird" / "dev_tables.json"
BIRD_DB_IDS = ({e["db_id"] for e in json.loads(
    _bird_tables_path.read_text(encoding="utf-8"))}
    if _bird_tables_path.exists() else set())

target_root = RAW_DIR / "bird" / "dev_databases"
target_root.mkdir(parents=True, exist_ok=True)

copied, found_names = 0, set()
if MINIDEV_SQLITE_ROOT.exists():
    for sqlite_path in MINIDEV_SQLITE_ROOT.rglob("*.sqlite"):
        db_id = sqlite_path.stem
        if BIRD_DB_IDS and db_id not in BIRD_DB_IDS:
            continue
        dest = target_root / db_id / f"{db_id}.sqlite"
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            shutil.copy(sqlite_path, dest)
            copied += 1
        found_names.add(db_id)
    print(f"Staged {copied} new BIRD SQLite DB(s); "
          f"present now: {len(found_names)}/{len(BIRD_DB_IDS)}.")
else:
    print("MINIDEV source folder not found - check the extraction location.")

# %% [markdown]
# ## Cell 10 - Verify (now reports gold_columns qualification rate)

# %%
def verify_dataset(dkey):
    qpath = DATA_DIR / dkey / "questions.json"
    spath = DATA_DIR / dkey / "schema.json"
    if not qpath.exists():
        print(f"[{dkey}] not built.")
        return None

    questions = json.loads(qpath.read_text(encoding="utf-8"))
    schema = json.loads(spath.read_text(encoding="utf-8")) if spath.exists() else []
    db_dir = DATA_DIR / dkey / "db"
    n_db_files = len(list(db_dir.glob("*.sqlite"))) if db_dir.exists() else 0

    df = pd.DataFrame(questions)
    if "gold_columns" in df.columns:
        df["n_gc"] = df["gold_columns"].apply(len)
        # fraction of gold columns that are qualified (contain a dot)
        def qual_rate(cols):
            if not cols:
                return np.nan
            return np.mean([("." in c) for c in cols])
        df["qual_rate"] = df["gold_columns"].apply(qual_rate)
        nonempty = (df["n_gc"] > 0).mean()
        mean_qual = df["qual_rate"].mean(skipna=True)
        df["n_join_gold"] = (df["gold_tables"].apply(len) - 1).clip(lower=0)
        dist = df["n_join_gold"].value_counts().sort_index()
    else:
        nonempty, mean_qual, dist = 0.0, np.nan, {}

    print(f"[{dkey}] questions={len(df)}, databases(schema)={len(schema)}, "
          f"sqlite_files={n_db_files}, gold_cols_nonempty={nonempty:.0%}, "
          f"gold_cols_qualified={mean_qual:.0%}")
    if len(dist):
        print(f"        JOIN-count distribution: {dict(dist)}")
    return df


for dk in ["D1", "D2", "D2_fin"]:
    verify_dataset(dk)


# %% [markdown]
# ## Cell 11 - Next step
# Re-run notebook 01 (contexts rebuild on the new gold columns), then notebook 02.

# %%
print("Dataset preparation complete (schema-qualified gold columns). "
      "Re-run notebook 01, then notebook 02.")

Core libraries imported.
Project directories ready.
Environment loaded. Model tiers:
  WEAK   -> gpt-4o-mini
  MID    -> gpt-5.4-mini
  STRONG -> gpt-5.4
Conditions: ['C1', 'C2', 'C3', 'C4', 'C5']
Datasets   : ['D1', 'D2', 'D2_fin', 'D3']
Grayscale plotting style configured.
Condition styles: {'C1': '0.15', 'C2': '0.3', 'C3': '0.45', 'C4': '0.6', 'C5': '0.75'}
Live check skipped (set RUN_LIVE_CHECK=True to test credentials).
Setup complete. Config snapshot saved.
Notebook 00b dependencies ready.
  [spider] schema tables=166, sqlite present/needed=20/20
[D1] wrote 1034 questions across 166 databases; copied 20 sqlite DB(s).
  [bird] schema tables=11, sqlite present=11
[D2] wrote 500 questions across 11 databases; copied 11 sqlite DB(s).
[D2_fin] wrote 32 questions across 1 databases; copied 1 sqlite DB(s).
Staged 0 new BIRD SQLite DB(s); present now: 11/11.
[D1] questions=1034, databases(schema)=166, sqlite_files=20, gold_cols_nonempty=96%, gold_cols_qualified=92%
        JOIN-count dis